# Compliance Checkpoint — FCA / MiFID II Human-in-the-Loop

**Addresses [#7687](https://github.com/langchain-ai/langgraph/issues/7687)**

A regulated payment-approval workflow built with LangGraph. It shows how to
combine `interrupt()` / `Command(resume=...)` human-in-the-loop review with a
tamper-evident, **append-only** compliance audit trail suitable for
FCA SYSC 3.2 record-keeping and MiFID II Art. 16 governance requirements.

**Graph:** `analyse -> compliance_gate -> [human_review] -> finalise`

* **analyse** - a mock model returns a confidence score and a risk class.
* **compliance_gate**
  * `risk == CRITICAL` -> hard block, the request never reaches a human
    (e.g. a sanctioned counterparty under FCA rules).
  * `confidence < 0.70` **or** `risk == HIGH` -> `interrupt()` for human review.
  * otherwise -> pass straight through to settlement.
* **human_review** - `interrupt()` pauses the run; `Command(resume=decision)`
  resumes it with `approve` / `reject` / `escalate`.
* **finalise** - executes only when the gate passed or a human approved.

**Audit design (per Igor Ganapolsky's comment on #7687):**

1. The **intent** to act is written to the audit log **before** the action
   executes, never after.
2. The **idempotency key lives on the business object** (`order_id`), *not* on
   the graph `run_id` - so replaying the same order in a new run does not
   double-write.
3. The audit store is **append-only**: `UPDATE` / `DELETE` are rejected at the
   database layer via triggers that `RAISE(ABORT)`.

No external API keys are required - the model is a deterministic mock.

In [ ]:
%pip install -qU langgraph langgraph-checkpoint-sqlite

In [ ]:
import sqlite3
import uuid
from datetime import datetime, timezone
from typing import Literal

from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

In [ ]:
# --- Business & policy constants -------------------------------------------
POLICY_VERSION = "FCA-SYSC-2024.1 / MiFID-II-RTS-6"
CONFIDENCE_THRESHOLD = 0.70


# --- Graph state ------------------------------------------------------------
class ComplianceState(TypedDict):
    request: str
    order_id: str  # the business object the idempotency key lives on
    idempotency_key: str  # uuid5(order_id) - stable across graph runs
    confidence: float
    risk_class: Literal["LOW", "HIGH", "CRITICAL"]
    decision: str | None
    reviewer: str | None
    audit_id: str | None
    outcome: str | None


# --- Append-only compliance audit store ------------------------------------
# A dedicated SQLite database, separate from the LangGraph checkpointer, that
# records WHAT the system intended to do and WHAT actually happened. Writes are
# append-only: UPDATE / DELETE are rejected at the database layer via triggers,
# so the trail is tamper-evident (FCA SYSC 3.2 / MiFID II Art. 16).
AUDIT = sqlite3.connect(":memory:", check_same_thread=False)

AUDIT.executescript(
    """
    CREATE TABLE IF NOT EXISTS audit_log (
        id                INTEGER PRIMARY KEY AUTOINCREMENT,
        event_type        TEXT NOT NULL CHECK (event_type IN ('intent', 'outcome')),
        graph_run_id      TEXT NOT NULL,
        checkpoint_id     TEXT,
        proposed_action   TEXT NOT NULL,
        risk_class        TEXT NOT NULL,
        policy_version    TEXT NOT NULL,
        decision          TEXT,
        reviewer_source   TEXT,
        timestamp_intent  TEXT,
        timestamp_outcome TEXT,
        outcome_ref       TEXT,
        idempotency_key   TEXT NOT NULL
    );

    -- Idempotency lives on the business object (idempotency_key), NOT the graph
    -- run id: replaying the same order in a new run must not double-write.
    CREATE UNIQUE INDEX IF NOT EXISTS ux_audit_business
        ON audit_log (idempotency_key, event_type);

    -- Append-only guarantees: any mutation attempt aborts the transaction.
    CREATE TRIGGER IF NOT EXISTS audit_block_update
    BEFORE UPDATE ON audit_log
    BEGIN
        SELECT RAISE(ABORT, 'audit_log is append-only: UPDATE forbidden');
    END;

    CREATE TRIGGER IF NOT EXISTS audit_block_delete
    BEFORE DELETE ON audit_log
    BEGIN
        SELECT RAISE(ABORT, 'audit_log is append-only: DELETE forbidden');
    END;
    """
)


def _now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def _proposed_action(state: ComplianceState) -> str:
    return f"EXECUTE_PAYMENT::{state['order_id']}"


def business_idempotency_key(order_id: str) -> str:
    """Deterministic per business object, and stable across graph runs."""
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"order:{order_id}"))


def log_intent(
    state: ComplianceState, config: RunnableConfig, proposed_action: str
) -> str:
    """Record the intent to act BEFORE the action executes (issue #7687)."""
    cfg = config["configurable"]
    cur = AUDIT.execute(
        """
        INSERT OR IGNORE INTO audit_log (
            event_type, graph_run_id, checkpoint_id, proposed_action,
            risk_class, policy_version, timestamp_intent, idempotency_key
        ) VALUES ('intent', ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            cfg.get("thread_id", ""),
            cfg.get("checkpoint_id", ""),
            proposed_action,
            state["risk_class"],
            POLICY_VERSION,
            _now(),
            state["idempotency_key"],
        ),
    )
    AUDIT.commit()
    return str(cur.lastrowid)

In [ ]:
def mock_llm(request: str) -> tuple[float, str]:
    """Deterministic stand-in for a real model - no API key required."""
    text = request.lower()
    if "sanction" in text or "ofsi" in text or "ofac" in text:
        return 0.42, "CRITICAL"
    if "offshore" in text or "unverified" in text or "new payee" in text:
        return 0.55, "HIGH"
    return 0.85, "LOW"


def analyse(state: ComplianceState, config: RunnableConfig) -> dict:
    confidence, risk_class = mock_llm(state["request"])
    enriched = {**state, "confidence": confidence, "risk_class": risk_class}
    # Intent is written HERE - before the gate routes and before any action runs.
    audit_id = log_intent(enriched, config, _proposed_action(state))
    return {
        "confidence": confidence,
        "risk_class": risk_class,
        "audit_id": audit_id,
    }

In [ ]:
def compliance_gate(state: ComplianceState, config: RunnableConfig) -> dict:
    risk = state["risk_class"]

    # CRITICAL: hard regulatory block. Never reaches a human (FCA sanctions).
    if risk == "CRITICAL":
        log_outcome(
            state,
            config,
            decision="blocked",
            reviewer="fca_policy_engine",
            outcome_ref="FCA_HARD_BLOCK",
        )
        return {
            "decision": "blocked",
            "reviewer": "fca_policy_engine",
            "outcome": "BLOCKED::FCA_SANCTIONS_LIST",
        }

    # Low confidence OR elevated risk: route to a human compliance officer.
    if state["confidence"] < CONFIDENCE_THRESHOLD or risk == "HIGH":
        return {"decision": "pending_human_review"}

    # Otherwise: automatically approved, straight to settlement.
    return {"decision": "auto_approved", "reviewer": "fca_policy_engine"}


def route_after_gate(state: ComplianceState) -> str:
    if state["risk_class"] == "CRITICAL":
        return "block"
    if state["decision"] == "pending_human_review":
        return "human_review"
    return "finalise"

In [ ]:
def human_review(state: ComplianceState, config: RunnableConfig) -> dict:
    # interrupt() pauses the graph and surfaces this payload to the operator.
    review = interrupt(
        {
            "prompt": "Compliance review required (MiFID II Art. 16 / FCA SYSC).",
            "request": state["request"],
            "confidence": state["confidence"],
            "risk_class": state["risk_class"],
            "options": ["approve", "reject", "escalate"],
        }
    )
    decision = review["decision"] if isinstance(review, dict) else str(review)
    reviewer = "human_compliance_officer"

    # approve -> continue to finalise; reject / escalate -> stop and record now.
    if decision != "approve":
        log_outcome(
            state,
            config,
            decision=decision,
            reviewer=reviewer,
            outcome_ref=f"HUMAN_{decision.upper()}",
        )
        return {
            "decision": decision,
            "reviewer": reviewer,
            "outcome": f"STOPPED::{decision.upper()}",
        }
    return {"decision": decision, "reviewer": reviewer}


def route_after_review(state: ComplianceState) -> str:
    return "finalise" if state["decision"] == "approve" else "stop"

In [ ]:
# Toggle used by scenario 5 to simulate a failure AFTER the intent is recorded.
INJECT_FINALISE_ERROR = False


def log_outcome(
    state: ComplianceState,
    config: RunnableConfig,
    decision: str,
    reviewer: str,
    outcome_ref: str,
) -> None:
    """Record the outcome after the action resolves - a separate append-only row."""
    cfg = config["configurable"]
    AUDIT.execute(
        """
        INSERT OR IGNORE INTO audit_log (
            event_type, graph_run_id, checkpoint_id, proposed_action,
            risk_class, policy_version, decision, reviewer_source,
            timestamp_outcome, outcome_ref, idempotency_key
        ) VALUES ('outcome', ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            cfg.get("thread_id", ""),
            cfg.get("checkpoint_id", ""),
            _proposed_action(state),
            state["risk_class"],
            POLICY_VERSION,
            decision,
            reviewer,
            _now(),
            outcome_ref,
            state["idempotency_key"],
        ),
    )
    AUDIT.commit()


def finalise(state: ComplianceState, config: RunnableConfig) -> dict:
    order_ref = f"SETTLE-{state['order_id']}"

    # ---- The real-world business action executes at this point ----
    if INJECT_FINALISE_ERROR:
        raise RuntimeError(
            "settlement gateway returned 504 - action failed AFTER intent was persisted"
        )

    log_outcome(
        state,
        config,
        decision=state.get("decision") or "auto_approved",
        reviewer=state.get("reviewer") or "fca_policy_engine",
        outcome_ref=order_ref,
    )
    return {"outcome": f"SETTLED::{order_ref}"}

In [ ]:
builder = StateGraph(ComplianceState)
builder.add_node("analyse", analyse)
builder.add_node("compliance_gate", compliance_gate)
builder.add_node("human_review", human_review)
builder.add_node("finalise", finalise)

builder.add_edge(START, "analyse")
builder.add_edge("analyse", "compliance_gate")
builder.add_conditional_edges(
    "compliance_gate",
    route_after_gate,
    {"human_review": "human_review", "finalise": "finalise", "block": END},
)
builder.add_conditional_edges(
    "human_review",
    route_after_review,
    {"finalise": "finalise", "stop": END},
)
builder.add_edge("finalise", END)

graph = builder.compile(checkpointer=MemorySaver())


def new_case(request: str, order_id: str) -> tuple[dict, dict]:
    """Build the initial state and a fresh graph config for one business order."""
    state = {
        "request": request,
        "order_id": order_id,
        "idempotency_key": business_idempotency_key(order_id),
    }
    # A new run id every time; the idempotency key stays pinned to the order.
    config = {"configurable": {"thread_id": f"run::{order_id}::{uuid.uuid4().hex[:8]}"}}
    return state, config


def show_audit(order_id: str) -> None:
    key = business_idempotency_key(order_id)
    rows = AUDIT.execute(
        """
        SELECT event_type, proposed_action, risk_class, decision,
               reviewer_source, outcome_ref
        FROM audit_log WHERE idempotency_key = ? ORDER BY id
        """,
        (key,),
    ).fetchall()
    print(f"audit trail for {order_id} (idempotency_key={key[:8]}...):")
    for r in rows:
        print(
            f"  [{r[0]:>7}] action={r[1]}  risk={r[2]}  "
            f"decision={r[3]}  reviewer={r[4]}  ref={r[5]}"
        )
    if not rows:
        print("  (no audit rows)")


# Offline Mermaid rendering of the workflow.
print(graph.get_graph().draw_mermaid())

## Scenario 1 - confidence 0.85, LOW -> auto-approved (no interrupt)

A routine payment to a verified payee. The gate lets it through automatically
and it settles without any human involvement.

In [ ]:
INJECT_FINALISE_ERROR = False
state, config = new_case("Wire GBP 5,000 to a verified UK payee", "ORD-1001")

result = graph.invoke(state, config)

assert "__interrupt__" not in result, "a low-risk request must not pause"
print("interrupted:", "__interrupt__" in result)
print("confidence :", result["confidence"], "| risk:", result["risk_class"])
print("decision   :", result["decision"])
print("outcome    :", result["outcome"])
show_audit("ORD-1001")

## Scenario 2 - confidence 0.55, HIGH -> interrupt -> human approves -> finalised

An elevated-risk payment. The graph pauses at `human_review`; the compliance
officer resumes it with `Command(resume="approve")` and it settles.

In [ ]:
INJECT_FINALISE_ERROR = False
state, config = new_case(
    "Wire GBP 90,000 to a new payee via an offshore account", "ORD-2002"
)

paused = graph.invoke(state, config)
print("paused for review:", "__interrupt__" in paused)
print("interrupt prompt :", paused["__interrupt__"][0].value["prompt"])

resumed = graph.invoke(Command(resume="approve"), config)
print("decision :", resumed["decision"], "| reviewer:", resumed["reviewer"])
print("outcome  :", resumed["outcome"])
assert resumed["outcome"].startswith("SETTLED")
show_audit("ORD-2002")

## Scenario 3 - confidence 0.55, HIGH -> interrupt -> human rejects -> stopped

The same elevated-risk payment, but this time the reviewer resumes with
`Command(resume="reject")`. The action is stopped and never settles.

In [ ]:
INJECT_FINALISE_ERROR = False
state, config = new_case(
    "Wire GBP 90,000 to a new payee via an offshore account", "ORD-3003"
)

paused = graph.invoke(state, config)
print("paused for review:", "__interrupt__" in paused)

resumed = graph.invoke(Command(resume="reject"), config)
print("decision :", resumed["decision"], "| reviewer:", resumed["reviewer"])
print("outcome  :", resumed["outcome"])
assert resumed["outcome"].startswith("STOPPED"), "a rejected action must not settle"
show_audit("ORD-3003")

## Scenario 4 - CRITICAL -> hard block at the gate (human never triggered)

A payment to a sanctioned counterparty. The gate blocks it outright; no
interrupt is raised and no human is ever consulted.

In [ ]:
INJECT_FINALISE_ERROR = False
state, config = new_case(
    "Wire funds to an entity on the OFSI sanctions list", "ORD-4004"
)

result = graph.invoke(state, config)
print("interrupted:", "__interrupt__" in result)  # the human is NEVER consulted
print("risk       :", result["risk_class"])
print("decision   :", result["decision"])
print("outcome    :", result["outcome"])
assert "__interrupt__" not in result and result["decision"] == "blocked"
show_audit("ORD-4004")

## Scenario 5 - node error post-execution -> intent was recorded *before* the error

`finalise` raises after the intent has already been persisted. The run fails,
but the audit trail still contains the `intent` row and no `outcome` row -
demonstrating that intent is durable even when the action fails (issue #7687).

In [ ]:
INJECT_FINALISE_ERROR = True
state, config = new_case("Wire GBP 4,000 to a verified UK payee", "ORD-5005")

try:
    graph.invoke(state, config)
except RuntimeError as exc:
    print("node raised:", exc)
finally:
    INJECT_FINALISE_ERROR = False

# The action failed, yet the intent row is durable and no outcome was written.
show_audit("ORD-5005")
events = [
    r[0]
    for r in AUDIT.execute(
        "SELECT event_type FROM audit_log WHERE idempotency_key = ?",
        (business_idempotency_key("ORD-5005"),),
    ).fetchall()
]
print("events recorded:", events)
assert "intent" in events and "outcome" not in events, (
    "intent must survive a failed action"
)

## Summary - the full append-only audit trail

Every business action leaves an `intent` row (written before it ran) and, when
it resolved, an `outcome` row. The store rejects any attempt to mutate history.

In [ ]:
print("=== Compliance audit trail (append-only) ===\n")
header = (
    f"{'order':10} {'event':8} {'risk':9} {'decision':18} {'reviewer':24} {'ref':22}"
)
print(header)
print("-" * len(header))
for row in AUDIT.execute(
    """
    SELECT event_type, risk_class, decision, reviewer_source,
           outcome_ref, proposed_action
    FROM audit_log ORDER BY id
    """
).fetchall():
    order = row[5].split("::")[-1]
    print(
        f"{order:10} {row[0]:8} {row[1]:9} "
        f"{str(row[2]):18} {str(row[3]):24} {str(row[4]):22}"
    )

# Tamper-evidence: the database itself rejects any mutation of the trail.
print("\nappend-only enforcement:")
try:
    AUDIT.execute("UPDATE audit_log SET decision = 'tampered' WHERE id = 1")
except sqlite3.Error as exc:
    print("  UPDATE ->", exc)
try:
    AUDIT.execute("DELETE FROM audit_log WHERE id = 1")
except sqlite3.Error as exc:
    print("  DELETE ->", exc)